# 01 - Coleta e Filtragem do Shapefile de Setores Censitários

Extrai o shapefile do IBGE (Censo 2022), filtra pelos setores de Recife e salva em GeoJSON.

In [ ]:
import zipfile
import os
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Caminhos base do projeto
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
ZIP_PATH   = os.path.join(ROOT, 'data', 'raw', 'setores_ibge', 'PE_setores_CD2022.zip')
EXTRACT_DIR = os.path.join(ROOT, 'data', 'raw', 'setores_ibge', 'PE_setores_CD2022')
OUTPUT_PATH = os.path.join(ROOT, 'data', 'raw', 'setores_ibge', 'recife_setores.geojson')

os.makedirs(EXTRACT_DIR, exist_ok=True)
print(f'Diretório de extração: {EXTRACT_DIR}')

In [ ]:
# Extrai o arquivo ZIP com os setores censitários de Pernambuco
print('Extraindo ZIP...')
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f'Arquivos extraídos em: {EXTRACT_DIR}')
    print('Conteúdo:', os.listdir(EXTRACT_DIR))
except FileNotFoundError:
    print(f'ERRO: arquivo não encontrado em {ZIP_PATH}')
    raise
except zipfile.BadZipFile:
    print('ERRO: arquivo ZIP inválido ou corrompido')
    raise

In [ ]:
# Localiza o arquivo .shp dentro do diretório extraído (busca recursiva)
shp_files = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.endswith('.shp'):
            shp_files.append(os.path.join(root, f))

if not shp_files:
    raise FileNotFoundError('Nenhum arquivo .shp encontrado após extração.')

SHP_PATH = shp_files[0]
print(f'Shapefile encontrado: {SHP_PATH}')

In [ ]:
# Carrega o shapefile com geopandas
print('Carregando shapefile...')
try:
    gdf = gpd.read_file(SHP_PATH)
    print(f'Total de setores em PE: {len(gdf)}')
    print('Colunas:', gdf.columns.tolist())
    print('CRS:', gdf.crs)
    display(gdf.head(3))
except Exception as e:
    print(f'ERRO ao carregar shapefile: {e}')
    raise

In [ ]:
# Filtra apenas os setores de Recife pelo código do município (CD_MUN = '2611606')
COD_RECIFE = '2611606'
print(f'Filtrando setores de Recife (CD_MUN == {COD_RECIFE})...')

recife = gdf[gdf['CD_MUN'] == COD_RECIFE].copy()
print(f'Setores de Recife encontrados: {len(recife)}')

In [ ]:
# Plota o mapa dos setores censitários de Recife
print('Gerando mapa...')
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
recife.plot(ax=ax, edgecolor='black', linewidth=0.3, color='steelblue', alpha=0.6)
ax.set_title('Setores Censitários de Recife - Censo 2022', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()
print('Mapa plotado com sucesso.')

In [ ]:
# Salva o GeoDataFrame filtrado como GeoJSON
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
print(f'Salvando GeoJSON em: {OUTPUT_PATH}')
try:
    recife.to_file(OUTPUT_PATH, driver='GeoJSON')
    print('Arquivo salvo com sucesso.')
except Exception as e:
    print(f'ERRO ao salvar GeoJSON: {e}')
    raise